# Step 11 – Cascade Round 2 (ASR → MT, Round 1 বনাম Round 2 ASR মডেল)
**Runtime → Change runtime type → GPU (T4 / L4 / A100 যেকোনো একটা)**, তারপর **Run all**।

- নতুন ASR চালাতে হয় না: Step 10 (turbo) আর Step 10b (small) যে `predictions_r1_*.csv` / `predictions_r2_*.csv` লিখেছে, সেগুলো নিয়ে **একই NLLB মডেল** (Step 3-এর `nllb-sylheti/final`, কোনো পরিবর্তন নেই) দিয়ে বাংলা আর ইংরেজিতে অনুবাদ হয়।
- সেটিং Step 3b-এর মতোই: একই `clean()`, beam 4, max 256, sacrebleu BLEU / chrF।
- MT-র শুধু **test split**-এর বাক্য স্কোর হয় (MT মডেল যেগুলো ট্রেনিংয়ে দেখেনি)। `test_new_all` এই কারণে বাদ।
- ফল সেভ হয় `Sylheti_Thesis/sylheti_mt/cascade_round2`-এ। মাঝপথে থেমে গেলে আবার Run all দিলে আগের অনুবাদ থেকে চলবে।
- প্রথমবার library বদলালে session নিজে restart হবে ("session crashed" দেখাবে) — তখন আবার **Run all**।
- কিছু মোছা হয় না; শেষ cell Drive-এ সব লিখে unmount করে।

## 1. Library আর Drive

In [ ]:
import os, sys, subprocess, time
from importlib.metadata import version, PackageNotFoundError
NEED = {"transformers": "4.48.3", "sacrebleu": "2.4.3"}
def _ver(p):
    try:
        return version(p)
    except PackageNotFoundError:
        return None
wrong = [p for p, v in NEED.items() if _ver(p) != v]
if wrong:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"] +
                   [p + "==" + v for p, v in NEED.items()], check=True)
    print("installed", wrong, "-> restarting the session; press Run all again")
    os.kill(os.getpid(), 9)

# আগের কোনো রান থেকে /content/drive লোকাল ফোল্ডার হয়ে থাকলে সেটা সরিয়ে রাখি (মুছি না), তারপর mount
if os.path.exists("/content/drive") and not os.path.ismount("/content/drive") and os.listdir("/content/drive"):
    new = "/content/drive_local_leftover_" + time.strftime("%H%M%S")
    os.rename("/content/drive", new); print("leftover local folder moved to", new)
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
import torch
assert torch.cuda.is_available(), "GPU নেই: Runtime -> Change runtime type -> GPU"
print(torch.cuda.get_device_name(0), "| transformers", version("transformers"), "| sacrebleu", version("sacrebleu"))

## 2. কোন ASR ফাইল অনুবাদ হবে

In [ ]:
import re, unicodedata, pandas as pd
ROOT   = "/content/drive/MyDrive/Sylheti_Thesis/Ultimate Thisis Data"
ASR_DIR = "/content/drive/MyDrive/Sylheti_Thesis/sylheti_asr"
MT_DIR  = "/content/drive/MyDrive/Sylheti_Thesis/sylheti_mt"
MT_MODEL = MT_DIR + "/nllb-sylheti/final"
OUT = MT_DIR + "/cascade_round2"; os.makedirs(OUT, exist_ok=True)
assert os.path.isdir(MT_MODEL), "MT মডেল পাওয়া যায়নি: " + MT_MODEL

# (ASR model, training round) -> prediction file (missing ones are skipped)
MODELS = {"turbo": ASR_DIR + "/whisper-large-v3-turbo-round2",
          "small": ASR_DIR + "/whisper-small-round2"}
ROUNDS = ["r1", "r2"]                       # r1 = trained on Round 1 data, r2 = Round 1 + Round 2 data
TEST_SETS = ["test_seen", "test_unseen", "test_new", "test_r2_seen"]
LANGS = {"bn": ("bangla", "ben_Beng"), "en": ("english", "eng_Latn")}

def clean(s):                                # same cleaning as Step 3 / 3b
    s = unicodedata.normalize("NFC", str(s))
    s = re.sub(r"[\"\'“”‘’(),.।?!:;\-/*…\[\]|]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

corpus = pd.read_csv(ROOT + "/corpus.csv", encoding="utf-8-sig")
ref = corpus[["excel_row", "sylheti_asr", "bangla", "english", "split"]].drop_duplicates("excel_row")
mt_test = set(ref.loc[ref.split == "test", "excel_row"])

found = {}
for m, folder in MODELS.items():
    for r in ROUNDS:
        for ts in TEST_SETS:
            f = f"{folder}/predictions_{r}_{ts}.csv"
            if os.path.exists(f):
                found[(m, r, ts)] = f
            else:
                print("not found (skipped):", f)

# safety check: keep only sentences in the MT test split (MT never saw them in training)
print("\nwill translate:")
for (m, r, ts), f in found.items():
    d = pd.read_csv(f)
    sp = ref.set_index("excel_row").reindex(d.excel_row.unique())["split"].fillna("not in corpus").value_counts().to_dict()
    keep = d.excel_row.isin(mt_test).sum()
    print(f"  {m:5s} {r} {ts:13s} recordings {len(d):5d} | MT split of sentences {sp} | kept {keep}")

## 3. MT মডেল লোড

In [ ]:
import gc, sacrebleu
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tok = AutoTokenizer.from_pretrained(MT_MODEL, src_lang="ben_Beng")
mt = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL, torch_dtype=torch.float16).to("cuda").eval()

def translate(texts, lang, bs=32, beams=4):  # same settings as Step 3b
    out, i = [], 0
    bos = tok.convert_tokens_to_ids(lang)
    while i < len(texts):
        batch = [t if t else "-" for t in texts[i:i + bs]]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=256).to("cuda")
        n_new = min(256, enc["input_ids"].shape[1] * 2 + 10)
        try:
            with torch.no_grad():
                ids = mt.generate(**enc, forced_bos_token_id=bos, num_beams=beams, max_new_tokens=n_new)
        except torch.cuda.OutOfMemoryError:
            del enc; gc.collect(); torch.cuda.empty_cache()
            if bs == 1:
                raise
            bs = max(1, bs // 2); print("out of memory -> batch size", bs); continue
        out += tok.batch_decode(ids, skip_special_tokens=True)
        i += len(batch)
    return out

def translate_unique(texts, lang):           # identical inputs get identical output (beam search), so translate each once
    uniq = sorted(set(texts))
    mp = dict(zip(uniq, translate(uniq, lang)))
    return [mp[t] for t in texts]

def score(hyp, refs):
    return (round(sacrebleu.corpus_bleu(list(hyp), [list(refs)]).score, 2),
            round(sacrebleu.corpus_chrf(list(hyp), [list(refs)]).score, 2))
print("MT model ready:", MT_MODEL)

## 4. ASR লেখা → অনুবাদ (প্রতিটা ফাইল আলাদা সেভ হয়)

In [ ]:
from tqdm.auto import tqdm
def cache_path(m, r, ts):
    return f"{OUT}/cascade_{m}_{r}_{ts}.csv"

for (m, r, ts), f in tqdm(list(found.items())):
    c = cache_path(m, r, ts)
    if os.path.exists(c):
        continue
    a = pd.read_csv(f)[["audio_path", "speaker", "excel_row", "pred"]]
    a = a[a.excel_row.isin(mt_test)].merge(ref, on="excel_row", how="inner")
    a["asr"] = a.pred.fillna("").map(clean)
    a["gold"] = a.sylheti_asr.map(clean)
    for k, (col, code_) in LANGS.items():
        a["mt_" + k] = translate_unique(a.asr.tolist(), code_)
    a.to_csv(c + ".tmp", index=False, encoding="utf-8-sig"); os.replace(c + ".tmp", c)
    print("saved", os.path.basename(c), len(a))

## 5. Gold সিলেটি লেখা থেকে অনুবাদ (upper bound) আর ফলাফল টেবিল

In [ ]:
# gold text of all needed sentences, translated once per sentence (reuses Round 1 cache if present)
need = set()
for key in found:
    need |= set(pd.read_csv(cache_path(*key), encoding="utf-8-sig").excel_row)
gold_cache = OUT + "/gold_translations.csv"
old = MT_DIR + "/cascade_4spk/gold_translations.csv"
g = pd.read_csv(gold_cache, encoding="utf-8-sig") if os.path.exists(gold_cache) else (
    pd.read_csv(old, encoding="utf-8-sig") if os.path.exists(old) else pd.DataFrame(columns=["excel_row"]))
g = g[g.excel_row.isin(need)]
missing = ref[ref.excel_row.isin(need - set(g.excel_row))].copy()
if len(missing):
    missing["gold"] = missing.sylheti_asr.map(clean)
    for k, (col, code_) in LANGS.items():
        missing["mt_" + k] = translate(missing.gold.tolist(), code_)
    g = pd.concat([g, missing], ignore_index=True)
g = g[["excel_row", "mt_bn", "mt_en"]].drop_duplicates("excel_row")
g.to_csv(gold_cache, index=False, encoding="utf-8-sig")
print("gold sentences:", len(g), "| newly translated:", len(missing))

rows = []
for (m, r, ts) in found:
    a = pd.read_csv(cache_path(m, r, ts), encoding="utf-8-sig")
    for k, (col, _) in LANGS.items():
        b, ch = score(a["mt_" + k].fillna(""), a[col])
        rows.append({"input": f"{m} {r}", "test set": ts, "direction": "syl->" + k, "files": len(a), "BLEU": b, "chrF": ch})
done_sets = {}
for (m, r, ts) in found:                     # gold upper bound on the same recordings
    if ts in done_sets:
        continue
    a = pd.read_csv(cache_path(m, r, ts), encoding="utf-8-sig")[["excel_row", "bangla", "english"]]
    a = a.merge(g, on="excel_row", how="inner"); done_sets[ts] = True
    for k, (col, _) in LANGS.items():
        b, ch = score(a["mt_" + k].fillna(""), a[col])
        rows.append({"input": "gold text", "test set": ts, "direction": "syl->" + k, "files": len(a), "BLEU": b, "chrF": ch})

res = pd.DataFrame(rows)
res.to_csv(OUT + "/cascade_results.csv", index=False)
wide = res.pivot_table(index=["test set", "input"], columns="direction", values=["BLEU", "chrF"]).round(2)
wide.columns = [f"{d} {m}" for m, d in wide.columns]
order = [c for c in ["syl->bn BLEU", "syl->bn chrF", "syl->en BLEU", "syl->en chrF"] if c in wide.columns]
wide = wide[order].reindex(TEST_SETS, level=0)
files = res.drop_duplicates(["input", "test set"]).set_index(["test set", "input"]).files
wide.insert(0, "files", files.reindex(wide.index).values)
pd.set_option("display.width", 200)
print(wide.to_string())
print("\nsaved:", OUT + "/cascade_results.csv")

## 6. Paired bootstrap: Round 1 বনাম Round 2 ASR-এর cascade (BLEU / chrF)
ASR-এর মতোই: 10,000 resample, seed 13, একই রেকর্ডিং দুই মডেলে। Δ = Round 2 − Round 1 (ধনাত্মক = Round 2 ভালো)।

In [ ]:
import numpy as np
from sacrebleu.metrics import BLEU, CHRF
N_BOOT, SEED = 10_000, 13
METRICS = {"BLEU": BLEU(), "chrF": CHRF()}

def sent_stats(metric, hyp, refs):
    return np.array(metric._extract_corpus_statistics(list(hyp), [list(refs)]), dtype=np.int64)
def from_stats(metric, S):
    return metric._compute_score_from_stats(S.sum(0).tolist()).score

out = []
for m in MODELS:
    for ts in TEST_SETS:
        if (m, "r1", ts) not in found or (m, "r2", ts) not in found:
            continue
        a1 = pd.read_csv(cache_path(m, "r1", ts), encoding="utf-8-sig")
        a2 = pd.read_csv(cache_path(m, "r2", ts), encoding="utf-8-sig")
        j = a1.merge(a2[["audio_path", "mt_bn", "mt_en"]], on="audio_path", suffixes=("_1", "_2"))
        rng = np.random.default_rng(SEED)
        idx = rng.integers(0, len(j), size=(N_BOOT, len(j)))
        for k, (col, _) in LANGS.items():
            for name, met in METRICS.items():
                S1 = sent_stats(met, j[f"mt_{k}_1"].fillna(""), j[col])
                S2 = sent_stats(met, j[f"mt_{k}_2"].fillna(""), j[col])
                s1, s2 = from_stats(met, S1), from_stats(met, S2)
                # check against the normal sacrebleu score
                ref_s = (sacrebleu.corpus_bleu if name == "BLEU" else sacrebleu.corpus_chrf)(
                    list(j[f"mt_{k}_1"].fillna("")), [list(j[col])]).score
                assert abs(s1 - ref_s) < 0.01, (name, s1, ref_s)
                d = np.array([from_stats(met, S2[ii]) - from_stats(met, S1[ii]) for ii in idx])
                p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
                out.append({"ASR": m, "test set": ts, "direction": "syl->" + k, "metric": name, "files": len(j),
                            "Round 1": round(s1, 2), "Round 2": round(s2, 2), "Δ": round(s2 - s1, 2),
                            "95% CI": f"[{np.percentile(d, 2.5):.2f}, {np.percentile(d, 97.5):.2f}]", "p": round(p, 4)})
        print("done:", m, ts)
bs = pd.DataFrame(out)
bs.to_csv(OUT + "/bootstrap_cascade_r1_vs_r2.csv", index=False)
print(bs.to_string(index=False))
print("\nsaved:", OUT + "/bootstrap_cascade_r1_vs_r2.csv")

## 7. কয়েকটা উদাহরণ (turbo Round 2, test_new)

In [ ]:
key = ("turbo", "r2", "test_new") if ("turbo", "r2", "test_new") in found else next(iter(found))
d = pd.read_csv(cache_path(*key), encoding="utf-8-sig").sample(5, random_state=1)
for _, r in d.iterrows():
    print("সিলেটি (আসল):", r.gold); print("ASR লিখেছে  :", r.asr)
    print("বাংলা (আসল) :", r.bangla); print("বাংলা (cascade):", r.mt_bn)
    print("English (ref):", r.english); print("English (cascade):", r.mt_en); print()

## 8. Drive-এ সব লিখে unmount

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print("Drive flushed and unmounted. ফলাফল:", OUT)